# AlikeAudience Skill Test：用户采集数据探索

# 一、读取与概览

## 1.1 读取数据

In [ ]:
LANG = "zh"
from pathlib import Path
from bisect import bisect_right
from zoneinfo import ZoneInfo
import csv
import gzip
import io
import ipaddress
import json
import re
import zipfile

import holidays
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
from scipy.spatial import cKDTree
from shapely.geometry import shape
from timezonefinder import TimezoneFinder
from ua_parser import parse

ROOT = Path.cwd()
INPUT = ROOT / "alikeaudience_data_test.csv"
OUTPUT = ROOT / "outputs" / "simple"
CHARTS = OUTPUT / "charts"
OUTPUT.mkdir(parents=True, exist_ok=True)
CHARTS.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
if LANG == "zh":
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS"]
    plt.rcParams["axes.unicode_minus"] = False

## 1.2 数据类型、唯一性与空值


In [ ]:
# 将五个原始字段全部按字符串读取，并把空白文本视为空值。
data = pd.read_csv(INPUT, dtype="string", keep_default_na=False)
data = data.replace(r"^\s*$", pd.NA, regex=True)

expected = ["user_id", "timestamp", "lat_long", "ip_address", "user_agent"]
assert data.columns.tolist() == expected
source_values = data[expected].copy()

quality_summary = pd.DataFrame(
    {
        "column": data.columns,
        "dtype": [str(data[c].dtype) for c in data.columns],
        "missing": [int(data[c].isna().sum()) for c in data.columns],
        "unique": [int(data[c].nunique(dropna=True)) for c in data.columns],
    }
)
quality_summary["missing_rate"] = quality_summary["missing"] / len(data)
quality_summary.to_csv(
    OUTPUT / "quality_summary.csv", index=False, encoding="utf-8-sig"
)

assert data["user_id"].nunique(dropna=True) == len(data)
display(quality_summary)

# 二、数据扩充与特征工程

## 2.1 经纬度

### 2.1.1 Enrichment：坐标修正与地理信息补充

#### 坐标拆分与顺序修正

In [ ]:
# 字段说明写的是 lat/long，但实际范围表明第一项是经度。
parts = data["lat_long"].str.split(expand=True)
first = pd.to_numeric(parts[0], errors="coerce")
second = pd.to_numeric(parts[1], errors="coerce")

coordinate_ranges = pd.DataFrame(
    {
        "source_position": ["first", "second"],
        "minimum": [first.min(), second.min()],
        "maximum": [first.max(), second.max()],
    }
)
display(coordinate_ranges)

assert first.between(-180, 180).all() and not first.between(-90, 90).all()
assert second.between(-90, 90).all()
data["latitude"] = second.astype("Float64")
data["longitude"] = first.astype("Float64")

#### 洲与国家／地区匹配

使用空间索引寻找覆盖每个坐标点的国家边界；只有唯一匹配时才写入国家和洲。

In [ ]:
# 将每组不同坐标匹配到 Natural Earth 国家边界。
geojson = json.loads(
    (ROOT / "reference/geography/countries.geojson").read_text(encoding="utf-8")
)
features = geojson["features"]
geometries = [shapely.make_valid(shape(f["geometry"])) for f in features]
tree = shapely.STRtree(geometries)

coordinates = data[["longitude", "latitude"]].drop_duplicates().reset_index(drop=True)
pairs = tree.query(
    shapely.points(
        coordinates["longitude"].to_numpy(), coordinates["latitude"].to_numpy()
    ),
    predicate="covered_by",
)
polygon_matches = {}
for point_index, polygon_index in pairs.T:
    polygon_matches.setdefault(int(point_index), set()).add(int(polygon_index))

continent_zh = {
    "Africa": "非洲",
    "Asia": "亚洲",
    "Europe": "欧洲",
    "North America": "北美洲",
    "South America": "南美洲",
    "Oceania": "大洋洲",
    "Antarctica": "南极洲",
}
geo_rows = []
for row_index in range(len(coordinates)):
    candidates = polygon_matches.get(row_index, set())
    if len(candidates) != 1:
        geo_rows.append(
            {
                "loc_country_code": pd.NA,
                "loc_country_en": pd.NA,
                "loc_country_zh": pd.NA,
                "loc_continent_en": pd.NA,
                "loc_continent_zh": pd.NA,
            }
        )
        continue
    properties = features[next(iter(candidates))]["properties"]
    code = (
        properties.get("ISO_A2_EH")
        or properties.get("ISO_A2")
        or properties.get("WB_A2")
    )
    if code == "-99":
        code = properties.get("WB_A2")
    continent = properties.get("CONTINENT")
    geo_rows.append(
        {
            "loc_country_code": code,
            "loc_country_en": properties.get("NAME_EN") or properties.get("NAME"),
            "loc_country_zh": properties.get("NAME_ZH") or properties.get("NAME"),
            "loc_continent_en": continent,
            "loc_continent_zh": continent_zh.get(continent, continent),
        }
    )

coordinate_lookup = pd.concat([coordinates, pd.DataFrame(geo_rows)], axis=1)
coordinate_lookup.loc[
    coordinate_lookup["loc_country_code"].eq("TW"), "loc_country_zh"
] = "台湾地区"

#### 最近参考城市与时区

**加载城市参考数据**

In [ ]:
# 匹配同一国家内最近的 GeoNames 参考城市及其时区。
city_columns = [
    "geonameid",
    "name",
    "asciiname",
    "alternatenames",
    "city_latitude",
    "city_longitude",
    "feature_class",
    "feature_code",
    "country_code",
    "cc2",
    "admin1",
    "admin2",
    "admin3",
    "admin4",
    "population",
    "elevation",
    "dem",
    "timezone",
    "modification_date",
]
with zipfile.ZipFile(ROOT / "reference/geography/cities15000.zip") as archive:
    cities = pd.read_csv(
        io.BytesIO(archive.read("cities15000.txt")),
        sep="\t",
        names=city_columns,
        dtype={"country_code": "string"},
        low_memory=False,
    )

coordinate_lookup["loc_city_en"] = pd.NA
coordinate_lookup["loc_city_zh"] = pd.NA
coordinate_lookup["loc_timezone"] = pd.NA

city_zh_overrides = {
    "Jakarta": "雅加达",
    "Surabaya": "泗水",
    "Bandung": "万隆",
    "Semarang": "三宝垄",
    "Medan": "棉兰",
    "Makassar": "望加锡",
    "Pekanbaru": "北干巴鲁",
    "Palembang": "巨港",
    "Bekasi": "勿加泗",
    "Seoul": "首尔",
    "Busan": "釜山",
    "Incheon": "仁川",
    "Taipei": "台北",
    "Taichung": "台中",
    "Kaohsiung": "高雄",
    "Osaka": "大阪",
    "Tokyo": "东京",
    "Sapporo": "札幌",
    "Nagoya": "名古屋",
    "Fukuoka": "福冈",
    "Singapore": "新加坡",
    "Hong Kong": "香港",
    "Bangkok": "曼谷",
}

**匹配最近参考城市**

把经纬度转换为三维球面坐标，再用 KDTree 查找同一国家内最近的参考城市。

In [ ]:
for country_code, coordinate_part in coordinate_lookup.dropna(
    subset=["loc_country_code"]
).groupby("loc_country_code"):
    city_part = cities.loc[cities["country_code"].eq(country_code)].reset_index(
        drop=True
    )
    if city_part.empty:
        continue
    city_lat = np.radians(city_part["city_latitude"].to_numpy())
    city_lon = np.radians(city_part["city_longitude"].to_numpy())
    city_xyz = np.column_stack(
        (
            np.cos(city_lat) * np.cos(city_lon),
            np.cos(city_lat) * np.sin(city_lon),
            np.sin(city_lat),
        )
    )
    point_lat = np.radians(coordinate_part["latitude"].to_numpy())
    point_lon = np.radians(coordinate_part["longitude"].to_numpy())
    point_xyz = np.column_stack(
        (
            np.cos(point_lat) * np.cos(point_lon),
            np.cos(point_lat) * np.sin(point_lon),
            np.sin(point_lat),
        )
    )
    _, nearest_index = cKDTree(city_xyz).query(point_xyz)
    nearest = city_part.iloc[nearest_index].reset_index(drop=True)
    coordinate_lookup.loc[coordinate_part.index, "loc_city_en"] = nearest[
        "name"
    ].to_numpy()
    coordinate_lookup.loc[coordinate_part.index, "loc_timezone"] = nearest[
        "timezone"
    ].to_numpy()

    chinese_names = []
    for city_name, aliases in zip(nearest["name"], nearest["alternatenames"]):
        if city_name in city_zh_overrides:
            chinese_names.append(city_zh_overrides[city_name])
            continue
        alias_list = [] if pd.isna(aliases) else str(aliases).split(",")
        cjk_names = [
            a.strip()
            for a in alias_list
            if re.fullmatch(r"[\u3400-\u9fff]{2,8}(?:市)?", a.strip())
        ]
        # GeoNames cities15000 does not label alternate-name languages.
        # 有 CJK 别名时优先使用，否则保留英文名称。
        chinese_names.append(cjk_names[0] if cjk_names else city_name)
    coordinate_lookup.loc[coordinate_part.index, "loc_city_zh"] = chinese_names

# 最近城市没有时区时，用 TimezoneFinder 根据坐标补充。

**补充缺失时区**

In [ ]:
missing_timezone = coordinate_lookup["loc_timezone"].isna()
timezone_finder = TimezoneFinder(in_memory=True)
coordinate_lookup.loc[missing_timezone, "loc_timezone"] = [
    timezone_finder.timezone_at_land(lng=lon, lat=lat)
    for lon, lat in coordinate_lookup.loc[
        missing_timezone, ["longitude", "latitude"]
    ].itertuples(index=False, name=None)
]

#### 合并地理字段

In [ ]:
data = data.merge(
    coordinate_lookup,
    on=["longitude", "latitude"],
    how="left",
    validate="many_to_one",
    sort=False,
)

### 2.1.2 验证

In [ ]:
geo_validation = pd.DataFrame(
    {
        "check": [
            "row count preserved",
            "latitude within -90 to 90",
            "longitude within -180 to 180",
            "country matched",
            "timezone matched",
        ],
        "records": [
            len(data),
            int(data["latitude"].between(-90, 90).sum()),
            int(data["longitude"].between(-180, 180).sum()),
            int(data["loc_country_en"].notna().sum()),
            int(data["loc_timezone"].notna().sum()),
        ],
    }
)
assert len(data) == len(source_values)
assert data["latitude"].between(-90, 90).all()
assert data["longitude"].between(-180, 180).all()
display(coordinate_ranges)
display(geo_validation)

## 2.2 Timestamp

### 2.2.1 Enrichment

In [ ]:
data["timestamp_utc"] = pd.to_datetime(data["timestamp"], utc=True, errors="coerce")
assert data["timestamp_utc"].notna().all()
assert data["timestamp"].str.endswith("Z").all()

data["date"] = pd.Series(pd.NA, index=data.index, dtype="string")
data["hour"] = pd.Series(pd.NA, index=data.index, dtype="Int64")
data["weekday"] = pd.Series(pd.NA, index=data.index, dtype="string")

for timezone_name, row_index in (
    data.dropna(subset=["loc_timezone"]).groupby("loc_timezone").groups.items()
):
    local = data.loc[row_index, "timestamp_utc"].dt.tz_convert(ZoneInfo(timezone_name))
    data.loc[row_index, "date"] = local.dt.strftime("%Y-%m-%d").to_numpy()
    data.loc[row_index, "hour"] = local.dt.hour.to_numpy()
    data.loc[row_index, "weekday"] = local.dt.day_name().to_numpy()

#### 节日标记

In [ ]:
data["is_holiday"] = pd.Series(pd.NA, index=data.index, dtype="boolean")
years = sorted(data["timestamp_utc"].dt.year.unique())
for country_code, row_index in (
    data.dropna(subset=["loc_country_code", "date"])
    .groupby("loc_country_code")
    .groups.items()
):
    try:
        calendar = holidays.country_holidays(country_code, years=years, observed=True)
        dates = pd.to_datetime(data.loc[row_index, "date"]).dt.date
        data.loc[row_index, "is_holiday"] = [day in calendar for day in dates]
    except NotImplementedError:
        data.loc[row_index, "is_holiday"] = pd.NA

# 明确补充此前核验过的两个 2022 年 12 月官方节日例外。
data.loc[
    data["loc_country_code"].eq("PH") & data["date"].eq("2022-12-26"), "is_holiday"
] = True
data.loc[
    data["loc_country_code"].eq("HK") & data["date"].eq("2022-12-27"), "is_holiday"
] = True

### 2.2.2 验证

In [ ]:
time_validation = pd.DataFrame(
    {
        "check": [
            "source timestamp ends with Z",
            "timestamp parsed",
            "timezone available",
            "local date available",
            "holiday status available",
        ],
        "records": [
            int(data["timestamp"].str.endswith("Z").sum()),
            int(data["timestamp_utc"].notna().sum()),
            int(data["loc_timezone"].notna().sum()),
            int(data["date"].notna().sum()),
            int(data["is_holiday"].notna().sum()),
        ],
    }
)
assert data["timestamp"].str.endswith("Z").all()
assert data["timestamp_utc"].notna().all()
assert (
    data.loc[data["loc_timezone"].notna(), ["date", "hour", "weekday"]]
    .notna()
    .all()
    .all()
)
display(time_validation)

## 2.3 IP Address

### 2.3.1 Enrichment

In [ ]:
ip_rows = []
for raw_ip in data["ip_address"].drop_duplicates():
    try:
        address = ipaddress.ip_address(str(raw_ip).strip())
        ip_rows.append(
            {
                "ip_address": raw_ip,
                "ip_object": address,
                "ip_version": address.version,
                "ip_is_global": address.is_global,
                "ip_is_private": address.is_private,
                "ip_is_reserved": address.is_reserved,
                "ip_is_multicast": address.is_multicast,
                "ip_is_loopback": address.is_loopback,
                "ip_is_link_local": address.is_link_local,
                "ip_is_unspecified": address.is_unspecified,
            }
        )
    except ValueError:
        ip_rows.append({"ip_address": raw_ip, "ip_object": None, "ip_version": pd.NA})

ip_lookup = pd.DataFrame(ip_rows)
for column in [
    "ip_is_global",
    "ip_is_private",
    "ip_is_reserved",
    "ip_is_multicast",
    "ip_is_loopback",
    "ip_is_link_local",
    "ip_is_unspecified",
]:
    ip_lookup[column] = ip_lookup[column].astype("boolean")
ip_lookup["ip_version"] = ip_lookup["ip_version"].astype("Int64")

#### 历史 IP 国家／地区匹配

In [ ]:
country_ranges = {}
for version_number in [4, 6]:
    rows = []
    range_path = ROOT / f"reference/ip/history/dbip-country-ipv{version_number}.csv"
    with range_path.open(encoding="utf-8") as source:
        for start, end, country_code in csv.reader(source):
            start_number = int(ipaddress.ip_address(start))
            end_number = int(ipaddress.ip_address(end))
            if start_number <= end_number:
                rows.append((start_number, end_number, country_code))
    rows.sort()
    starts = [row[0] for row in rows]
    maximum_ends = []
    running_end = -1
    for _, end_number, _ in rows:
        running_end = max(running_end, end_number)
        maximum_ends.append(running_end)
    country_ranges[version_number] = (starts, rows, maximum_ends)

historical_country_codes = []
for address, version_number, is_global in ip_lookup[
    ["ip_object", "ip_version", "ip_is_global"]
].itertuples(index=False, name=None):
    if address is None or not bool(is_global):
        historical_country_codes.append(pd.NA)
        continue
    number = int(address)
    starts, rows, maximum_ends = country_ranges[int(version_number)]
    position = bisect_right(starts, number) - 1
    matches = set()
    while position >= 0 and maximum_ends[position] >= number:
        if rows[position][1] >= number:
            matches.add(rows[position][2])
        position -= 1
    matches -= {"", "XX", "ZZ"}
    historical_country_codes.append(next(iter(matches)) if len(matches) == 1 else pd.NA)

ip_lookup["ip_country_code"] = pd.Series(historical_country_codes, dtype="string")

#### 历史 ASN 匹配

In [ ]:
asn_prefixes = {}
for version_number, filename in {
    4: "routeviews-rv2-20221215-1200.pfx2as.gz",
    6: "routeviews-rv6-20221215-1200.pfx2as.gz",
}.items():
    groups = {}
    with gzip.open(ROOT / "reference/ip/history/asn" / filename, "rt") as source:
        for line in source:
            prefix, prefix_length, origin = line.split()
            network = ipaddress.ip_network(f"{prefix}/{prefix_length}", strict=True)
            groups.setdefault(network.prefixlen, {})[
                int(network.network_address)
            ] = origin
    asn_prefixes[version_number] = sorted(groups.items(), reverse=True)

asn_values = []
for address, version_number, is_global in ip_lookup[
    ["ip_object", "ip_version", "ip_is_global"]
].itertuples(index=False, name=None):
    if address is None or not bool(is_global):
        asn_values.append(pd.NA)
        continue
    number = int(address)
    origin = None
    for prefix_length, entries in asn_prefixes[int(version_number)]:
        shift = address.max_prefixlen - prefix_length
        origin = entries.get((number >> shift) << shift)
        if origin is not None:
            break
    asn_values.append(int(origin) if origin is not None and origin.isdigit() else pd.NA)

ip_lookup["ip_asn"] = pd.Series(asn_values, dtype="Int64")

In [ ]:
asn_to_organization_id = {}
asn_fallback_name = {}
organization_names = {}
record_format = None
with gzip.open(
    ROOT / "reference/ip/history/asn/20221001.as-org2info.txt.gz",
    "rt",
    encoding="utf-8",
    errors="replace",
) as source:
    for line in source:
        line = line.rstrip("\n")
        if line.startswith("# format:org_id|"):
            record_format = "organization"
            continue
        if line.startswith("# format:aut|"):
            record_format = "asn"
            continue
        if not line or line.startswith("#"):
            continue
        fields = line.split("|")
        if record_format == "organization" and len(fields) >= 3:
            organization_names[fields[0]] = fields[2]
        elif record_format == "asn" and len(fields) >= 4 and fields[0].isdigit():
            asn_to_organization_id[int(fields[0])] = fields[3]
            asn_fallback_name[int(fields[0])] = fields[2]

ip_lookup["ip_asn_organization"] = pd.Series(
    [
        (
            organization_names.get(
                asn_to_organization_id.get(int(asn)), asn_fallback_name.get(int(asn))
            )
            if pd.notna(asn)
            else pd.NA
        )
        for asn in ip_lookup["ip_asn"]
    ],
    dtype="string",
)
ip_lookup["ip_asn_name"] = pd.Series(
    [
        asn_fallback_name.get(int(asn)) if pd.notna(asn) else pd.NA
        for asn in ip_lookup["ip_asn"]
    ],
    dtype="string",
)

In [ ]:
country_name_en = {}
country_name_zh = {}
for feature in features:
    properties = feature["properties"]
    code = (
        properties.get("ISO_A2_EH")
        or properties.get("ISO_A2")
        or properties.get("WB_A2")
    )
    if code == "-99":
        code = properties.get("WB_A2")
    country_name_en[code] = properties.get("NAME_EN") or properties.get("NAME")
    country_name_zh[code] = properties.get("NAME_ZH") or properties.get("NAME")
country_name_zh["TW"] = "台湾地区"

ip_lookup["ip_country_en"] = (
    ip_lookup["ip_country_code"].map(country_name_en).astype("string")
)
ip_lookup["ip_country_zh"] = (
    ip_lookup["ip_country_code"].map(country_name_zh).astype("string")
)

data = data.merge(
    ip_lookup.drop(columns="ip_object"),
    on="ip_address",
    how="left",
    validate="many_to_one",
    sort=False,
)

### 2.3.2 验证

In [ ]:
comparable = data["loc_country_code"].notna() & data["ip_country_code"].notna()
same_country = data["loc_country_code"].eq(data["ip_country_code"]) & comparable
ip_location_agreement = pd.DataFrame(
    {
        "result": ["same", "different", "not_comparable"],
        "records": [
            int(same_country.sum()),
            int((comparable & ~same_country).sum()),
            int((~comparable).sum()),
        ],
    }
)

ip_validation = pd.DataFrame(
    {
        "check": [
            "IP parsed",
            "public IP",
            "historical country matched",
            "historical ASN matched",
            "historical ASN named",
            "historical ASN organization named",
            "country fields comparable",
        ],
        "records": [
            int(data["ip_version"].notna().sum()),
            int(data["ip_is_global"].sum()),
            int(data["ip_country_en"].notna().sum()),
            int(data["ip_asn"].notna().sum()),
            int(data["ip_asn_name"].notna().sum()),
            int(data["ip_asn_organization"].notna().sum()),
            int(comparable.sum()),
        ],
    }
)
assert data["ip_version"].notna().all()
assert (
    data.loc[
        ~data["ip_is_global"],
        [
            "ip_country_en",
            "ip_country_zh",
            "ip_asn",
            "ip_asn_name",
            "ip_asn_organization",
        ],
    ]
    .isna()
    .all()
    .all()
)
display(ip_validation)
display(ip_location_agreement)

## 2.4 User Agent

### 2.4.1 Enrichment

In [ ]:
ua_rows = []
for raw_ua in data["user_agent"].dropna().drop_duplicates():
    parsed = parse(raw_ua)
    os_parts = [
        getattr(parsed.os, part, None)
        for part in ["major", "minor", "patch", "patch_minor"]
    ]
    os_version = ".".join(str(part) for part in os_parts if part is not None) or pd.NA

    if raw_ua.startswith("Dalvik/"):
        engine = "Dalvik"
    elif "AppleWebKit/" in raw_ua:
        engine = "AppleWebKit"
    elif "Trident/" in raw_ua:
        engine = "Trident"
    elif "Gecko/" in raw_ua:
        engine = "Gecko"
    else:
        engine = "Other / unresolved"

    parsed_model = getattr(parsed.device, "model", None)
    parsed_brand = getattr(parsed.device, "brand", None)
    parsed_os = getattr(parsed.os, "family", None)
    device_model = parsed_model if parsed_model not in [None, "Other"] else pd.NA
    if pd.notna(device_model):
        device_model = re.split(r"\s+Build/", str(device_model), maxsplit=1)[0].strip()
        device_model = re.sub(r";\s*wv\s*$", "", device_model).strip()

    explicit_webview = bool(re.search(r"(?:[;(]\s*)wv(?:[;)])", raw_ua))
    dalvik = raw_ua.startswith("Dalvik/")
    ua_rows.append(
        {
            "user_agent": raw_ua,
            "ua_engine": engine,
            "os": parsed_os if parsed_os not in [None, "Other"] else pd.NA,
            "os_version": os_version,
            "device_brand": (
                parsed_brand if parsed_brand not in [None, "Other"] else pd.NA
            ),
            "device_model": device_model,
            "is_android_app": bool(
                "Android" in raw_ua and (explicit_webview or dalvik)
            ),
        }
    )

In [ ]:
ua_lookup = pd.DataFrame(ua_rows)
ua_lookup["is_android_app"] = ua_lookup["is_android_app"].astype("boolean")
data = data.merge(
    ua_lookup, on="user_agent", how="left", validate="many_to_one", sort=False
)
data["is_android_app"] = data["is_android_app"].astype("boolean")

### 2.4.2 验证

In [ ]:
ua_missing = data["user_agent"].isna()
ua_validation = pd.DataFrame(
    {
        "check": [
            "UA present",
            "UA missing but row retained",
            "OS resolved",
            "device brand resolved",
            "explicit Android app evidence",
        ],
        "records": [
            int((~ua_missing).sum()),
            int(ua_missing.sum()),
            int(data["os"].notna().sum()),
            int(data["device_brand"].notna().sum()),
            int(data["is_android_app"].eq(True).sum()),
        ],
    }
)
assert len(data) == len(source_values)
assert (
    data.loc[
        ua_missing,
        [
            "ua_engine",
            "os",
            "os_version",
            "device_brand",
            "device_model",
            "is_android_app",
        ],
    ]
    .isna()
    .all()
    .all()
)
display(ua_validation)

## 2.5 最终字段

In [ ]:
final_columns = [
    "user_id",
    "timestamp",
    "lat_long",
    "ip_address",
    "user_agent",
    "latitude",
    "longitude",
    "loc_continent_en",
    "loc_continent_zh",
    "loc_country_en",
    "loc_country_zh",
    "loc_city_en",
    "loc_city_zh",
    "loc_timezone",
    "timestamp_utc",
    "date",
    "hour",
    "weekday",
    "is_holiday",
    "ip_version",
    "ip_is_global",
    "ip_is_private",
    "ip_is_reserved",
    "ip_is_multicast",
    "ip_is_loopback",
    "ip_is_link_local",
    "ip_is_unspecified",
    "ip_country_en",
    "ip_country_zh",
    "ip_asn",
    "ip_asn_name",
    "ip_asn_organization",
    "ua_engine",
    "os",
    "os_version",
    "device_brand",
    "device_model",
    "is_android_app",
]
events = data[final_columns].copy()

In [ ]:
assert events.shape == (100000, 38)
assert events["user_id"].is_unique
pd.testing.assert_frame_equal(
    events[["user_id", "timestamp", "lat_long", "ip_address", "user_agent"]],
    source_values,
    check_dtype=False,
)

events.to_parquet(OUTPUT / "events_enriched.parquet", index=False)
written_events = pd.read_parquet(OUTPUT / "events_enriched.parquet")
pd.testing.assert_frame_equal(
    events.astype("string").fillna("<MISSING>"),
    written_events.astype("string").fillna("<MISSING>"),
)

metadata = {
    "records": len(events),
    "columns": len(events.columns),
    "raw_columns": 5,
    "coordinate_order": "source: longitude latitude; output: latitude longitude",
    "timestamp_basis": "UTC source converted to coordinate-derived local timezone",
    "ip_country_reference": "DB-IP country snapshot published 2022-12-02",
    "ip_asn_reference": "RouteViews prefix-to-origin snapshot 2022-12-15 12:00 UTC",
    "ip_asn_name_reference": "CAIDA AS Organizations 2022-10",
    "ip_lookup_filter": "Python ipaddress is_global == True",
    "ua_missing_policy": "retain row and leave UA-derived fields missing",
}
(OUTPUT / "metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)

display(
    pd.DataFrame(
        {
            "field": events.columns,
            "dtype": [str(events[c].dtype) for c in events.columns],
        }
    )
)
print(events.shape)

# 三、汇总与可视化

In [ ]:
suffix = "zh" if LANG == "zh" else "en"
colors = ["#177E89", "#D77A35", "#496DA4", "#76A56A", "#AB6C8E", "#6A6A6A"]

geo_charts = CHARTS / "geography"
time_charts = CHARTS / "time"
ip_charts = CHARTS / "ip"
ua_charts = CHARTS / "ua"
for chart_folder in [geo_charts, time_charts, ip_charts, ua_charts]:
    chart_folder.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        "font.size": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.unicode_minus": False,
    }
)


## 3.1 经纬度

### 3.1.1 汇总表

In [ ]:
geo_country = (
    data.groupby(["loc_country_en", "loc_country_zh"], dropna=False)
    .size()
    .rename("records")
    .reset_index()
    .sort_values("records", ascending=False)
)
geo_country["sample_share"] = geo_country["records"] / len(data)

geo_city = (
    data.groupby(
        ["loc_country_en", "loc_country_zh", "loc_city_en", "loc_city_zh"],
        dropna=False,
    )
    .size()
    .rename("records")
    .reset_index()
    .sort_values("records", ascending=False)
)

geo_daily_source = data.assign(
    event_date_utc=data["timestamp_utc"].dt.strftime("%Y-%m-%d"),
    market=data["loc_country_en"].fillna("Unmatched"),
)
geo_country_daily = (
    geo_daily_source.groupby(["event_date_utc", "market"])
    .size()
    .rename("records")
    .reset_index()
)

geo_os_source = data.dropna(subset=["loc_country_en"]).assign(
    os_group=data["os"].where(data["os"].isin(["Android", "iOS"]), "Other / unknown")
)
geo_country_os = (
    geo_os_source.groupby(["loc_country_en", "os_group"])
    .size()
    .rename("records")
    .reset_index()
)

country_name_rows = data.dropna(subset=["loc_country_en"]).drop_duplicates(
    "loc_country_en"
)
country_labels_en = dict(
    zip(country_name_rows["loc_country_en"], country_name_rows["loc_country_en"])
)
country_labels_zh = dict(
    zip(country_name_rows["loc_country_en"], country_name_rows["loc_country_zh"])
)
country_labels_en["Unmatched"] = "Unmatched"
country_labels_zh["Unmatched"] = "未匹配"
country_labels = country_labels_zh if LANG == "zh" else country_labels_en

for table_name, table in {
    "geo_country": geo_country,
    "geo_city": geo_city,
    "geo_country_daily": geo_country_daily,
    "geo_country_os": geo_country_os,
}.items():
    table.to_csv(OUTPUT / f"{table_name}.csv", index=False, encoding="utf-8-sig")

display(geo_country.head(10))


In [ ]:
import h3

h3_cells = pd.Series(
    [
        h3.latlng_to_cell(latitude, longitude, 5)
        for latitude, longitude in zip(data["latitude"], data["longitude"])
    ],
    name="h3_r5",
)
geo_spatial_cells = (
    h3_cells.value_counts().rename_axis("h3_r5").rename("records").reset_index()
)
geo_spatial_cells.to_csv(
    OUTPUT / "geo_spatial_cells.csv", index=False, encoding="utf-8-sig"
)


### 3.1.2 主要市场的附近城市

In [ ]:
selected_markets = ["Japan", "Indonesia", "South Korea"]
fig, axes = plt.subplots(figsize=(13, 7.5), ncols=3)
fig.subplots_adjust(left=0.13, right=0.96, top=0.86, bottom=0.13, wspace=0.72)
fig.suptitle(
    "主要市场的附近城市" if LANG == "zh" else "Nearby cities in major markets",
    fontsize=21,
    weight="bold",
    color="#17364B",
)

for ax, market, market_color in zip(axes, selected_markets, colors):
    city_part = (
        geo_city.loc[
            geo_city["loc_country_en"].eq(market) & geo_city["loc_city_en"].notna()
        ]
        .head(6)
        .sort_values("records")
    )
    city_labels = (
        city_part["loc_city_zh"].fillna(city_part["loc_city_en"])
        if LANG == "zh"
        else city_part["loc_city_en"]
    )
    ax.barh(city_labels, city_part["records"], color=market_color)
    ax.set_title(country_labels[market], weight="bold")
    ax.set_xlabel("记录数" if LANG == "zh" else "Records")
    limit = city_part["records"].max()
    ax.set_xlim(0, limit * 1.30)
    for row_number, record_count in enumerate(city_part["records"]):
        ax.text(
            record_count + limit * 0.025,
            row_number,
            f"{record_count:,}",
            va="center",
        )

fig.savefig(geo_charts / f"02_nearby_cities_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![主要市场的附近城市](outputs/simple/charts/geography/02_nearby_cities_zh.png)

### 3.1.3 各国家／地区每日记录数

In [ ]:
daily_pivot = geo_country_daily.pivot(
    index="event_date_utc", columns="market", values="records"
).fillna(0)
selected_markets = (
    geo_country.dropna(subset=["loc_country_en"]).head(5)["loc_country_en"].tolist()
)
other_markets = daily_pivot.drop(columns=selected_markets).sum(axis=1)
x = np.arange(len(daily_pivot))

fig, ax = plt.subplots(figsize=(13, 7.5))
fig.subplots_adjust(left=0.09, right=0.95, top=0.86, bottom=0.13)
fig.suptitle(
    "每日记录数（UTC）" if LANG == "zh" else "Daily records (UTC)",
    fontsize=21,
    weight="bold",
    color="#17364B",
)
values = [daily_pivot[market].to_numpy() for market in selected_markets]
values.append(other_markets.to_numpy())
labels = [country_labels[market] for market in selected_markets]
labels.append("其他及未匹配" if LANG == "zh" else "Other / unresolved")
ax.stackplot(x, *values, labels=labels, colors=colors)
ax.set_xticks(x[::3], [date[5:] for date in daily_pivot.index[::3]])
ax.set_xlim(0, len(x) - 1)
ax.set_ylabel("记录数" if LANG == "zh" else "Records")
ax.legend(loc="upper right", frameon=False, ncols=2)
fig.savefig(geo_charts / f"03_country_daily_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![每日记录数](outputs/simple/charts/geography/03_country_daily_zh.png)

### 3.1.4 主要市场的操作系统构成

In [ ]:
selected_country_rows = geo_country.loc[
    geo_country["loc_country_en"].notna() & geo_country["records"].ge(1000)
]
selected_markets = selected_country_rows["loc_country_en"].tolist()
os_pivot = (
    geo_country_os.pivot(index="loc_country_en", columns="os_group", values="records")
    .reindex(selected_markets)
    .fillna(0)
)
os_share = os_pivot.div(os_pivot.sum(axis=1), axis=0) * 100
y = np.arange(len(selected_markets))

fig, axes = plt.subplots(figsize=(13, 7.5), ncols=2)
fig.subplots_adjust(left=0.15, right=0.94, top=0.86, bottom=0.13, wspace=0.40)
fig.suptitle(
    (
        "主要市场的操作系统构成"
        if LANG == "zh"
        else "Operating-system composition by market"
    ),
    fontsize=21,
    weight="bold",
    color="#17364B",
)
left = np.zeros(len(selected_markets))
for os_group, group_color in zip(
    ["Android", "iOS", "Other / unknown"],
    [colors[0], colors[1], "#BBBBBB"],
):
    values = os_share[os_group].to_numpy()
    axes[0].barh(y, values, left=left, label=os_group, color=group_color)
    left += values

market_labels = [country_labels[market] for market in selected_markets]
for ax in axes:
    ax.set_yticks(y, market_labels)
    ax.invert_yaxis()
axes[0].set_xlim(0, 100)
axes[0].set_xlabel("市场内占比（%）" if LANG == "zh" else "Within-market share (%)")
axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncols=3, frameon=False)

ios_share = os_share["iOS"].to_numpy()
axes[1].scatter(ios_share, y, color=colors[1])
axes[1].set_xlim(0, max(6, ios_share.max() * 1.25))
axes[1].set_xlabel("iOS 占比（%）" if LANG == "zh" else "iOS share (%)")
for row_number, value in enumerate(ios_share):
    axes[1].text(value + 0.10, row_number, f"{value:.2f}%", va="center")

fig.savefig(geo_charts / f"04_country_os_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![主要市场操作系统构成](outputs/simple/charts/geography/04_country_os_zh.png)

### 3.1.5 地理分布图

In [ ]:
from matplotlib.collections import PolyCollection
from matplotlib.colors import LogNorm

shown_cells = geo_spatial_cells.loc[geo_spatial_cells["records"].ge(10)]
cell_polygons = [
    [(longitude, latitude) for latitude, longitude in h3.cell_to_boundary(cell)]
    for cell in shown_cells["h3_r5"]
]

fig, axes = plt.subplots(figsize=(13, 7.5), ncols=2)
fig.subplots_adjust(left=0.07, right=0.88, top=0.86, bottom=0.13, wspace=0.30)
fig.suptitle(
    "地理分布" if LANG == "zh" else "Geographic distribution",
    fontsize=21,
    weight="bold",
    color="#17364B",
)
map_bounds = [(93, 148, -13, 48), (105.6, 108.2, -7.5, -5.7)]
for ax, bounds in zip(axes, map_bounds):
    for geometry in geometries:
        for boundary in shapely.get_parts(shapely.boundary(geometry)):
            x, y = boundary.xy
            ax.plot(x, y, color="#A0ACA4", linewidth=0.30)
    collection = PolyCollection(
        cell_polygons,
        array=shown_cells["records"].to_numpy(float),
        cmap="YlOrRd",
        norm=LogNorm(10, shown_cells["records"].max()),
        linewidths=0,
    )
    ax.add_collection(collection)
    ax.set_xlim(bounds[:2])
    ax.set_ylim(bounds[2:])
    ax.set_aspect("equal")
    ax.set_facecolor("#F2F7FA")
    ax.set_xlabel("经度" if LANG == "zh" else "Longitude")
    ax.set_ylabel("纬度" if LANG == "zh" else "Latitude")

map_labels = [
    ("Japan", 139, 39),
    ("Indonesia", 111, -6),
    ("South Korea", 128, 36),
    ("Taiwan", 121, 24),
    ("Philippines", 124, 12),
    ("Thailand", 100, 16),
    ("Vietnam", 109, 18),
    ("Malaysia", 102, 4),
    ("Singapore", 104, 1),
]
for market, longitude, latitude in map_labels:
    axes[0].annotate(
        country_labels.get(market, market),
        (longitude, latitude),
        fontsize=8,
        bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "none"},
    )

city_annotations = [
    ("雅加达附近" if LANG == "zh" else "Near Jakarta", 106.867, -6.233, (-65, 30)),
    ("万隆附近" if LANG == "zh" else "Near Bandung", 107.615, -6.972, (10, -30)),
]
for label, longitude, latitude, offset in city_annotations:
    axes[1].annotate(
        label,
        (longitude, latitude),
        xytext=offset,
        textcoords="offset points",
        arrowprops={"arrowstyle": "->"},
        bbox={"facecolor": "white", "edgecolor": "#CCD4DA"},
    )
axes[1].set_title(
    "印度尼西亚 · 爪哇岛西部" if LANG == "zh" else "Indonesia: western Java"
)
colorbar_axis = fig.add_axes([0.92, 0.25, 0.015, 0.40])
fig.colorbar(
    collection,
    cax=colorbar_axis,
    label="记录数" if LANG == "zh" else "Records",
)
fig.savefig(geo_charts / f"05_geographic_map_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![地理分布图](outputs/simple/charts/geography/05_geographic_map_zh.png)

## 3.2 Timestamp

### 3.2.1 汇总表

In [ ]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
weekday_number = {name: number for number, name in enumerate(weekday_order)}
country_code_labels_en = {
    "JP": "Japan",
    "ID": "Indonesia",
    "KR": "South Korea",
    "VN": "Vietnam",
    "TH": "Thailand",
    "SG": "Singapore",
    "HK": "Hong Kong",
}
country_code_labels_zh = {
    "JP": "日本",
    "ID": "印度尼西亚",
    "KR": "韩国",
    "VN": "越南",
    "TH": "泰国",
    "SG": "新加坡",
    "HK": "香港",
}
country_code_labels = country_code_labels_zh if LANG == "zh" else country_code_labels_en

time_daily_local = (
    data.dropna(subset=["date"]).groupby("date").size().rename("records").reset_index()
)
time_daily_country = (
    data.dropna(subset=["loc_country_code", "date", "weekday"])
    .groupby(["loc_country_code", "date", "weekday"])
    .agg(records=("user_id", "size"), is_holiday=("is_holiday", "max"))
    .reset_index()
)
time_weekday = (
    time_daily_country.groupby(["loc_country_code", "weekday"])
    .agg(
        records=("records", "sum"),
        calendar_days=("date", "nunique"),
        daily_mean=("records", "mean"),
        daily_median=("records", "median"),
    )
    .reset_index()
)
time_weekday["weekday_number"] = time_weekday["weekday"].map(weekday_number)
time_weekday = time_weekday.sort_values(["loc_country_code", "weekday_number"])


In [ ]:
time_hourly = (
    data.dropna(subset=["loc_country_code", "hour"])
    .groupby(["loc_country_code", "hour"])
    .size()
    .rename("records")
    .reset_index()
)
time_hourly["country_hour_share"] = time_hourly["records"] / time_hourly.groupby(
    "loc_country_code"
)["records"].transform("sum")

time_weekday_hour = (
    data.dropna(subset=["loc_country_code", "weekday", "hour"])
    .groupby(["loc_country_code", "weekday", "hour"])
    .size()
    .rename("records")
    .reset_index()
)
weekday_days = (
    data.dropna(subset=["loc_country_code", "date", "weekday"])
    .groupby(["loc_country_code", "weekday"])["date"]
    .nunique()
    .rename("calendar_days")
    .reset_index()
)
time_weekday_hour = time_weekday_hour.merge(
    weekday_days, on=["loc_country_code", "weekday"], validate="many_to_one"
)
time_weekday_hour["daily_mean"] = (
    time_weekday_hour["records"] / time_weekday_hour["calendar_days"]
)
time_weekday_hour["weekday_number"] = time_weekday_hour["weekday"].map(weekday_number)

for table_name, table in {
    "time_daily_local": time_daily_local,
    "time_weekday": time_weekday,
    "time_hourly": time_hourly,
    "time_weekday_hour": time_weekday_hour,
}.items():
    table.to_csv(OUTPUT / f"{table_name}.csv", index=False, encoding="utf-8-sig")


In [ ]:
holiday_rows = []
first_date = pd.Timestamp(data["date"].dropna().min()).date()
last_date = pd.Timestamp(data["date"].dropna().max()).date()
holiday_years = sorted(data["timestamp_utc"].dt.year.unique())

for country_code in ["ID", "SG", "HK"]:
    calendar = holidays.country_holidays(
        country_code, years=holiday_years, observed=True
    )
    if country_code == "HK":
        calendar[pd.Timestamp("2022-12-27").date()] = (
            "Second weekday after Christmas Day"
        )
    for holiday_date, holiday_name in sorted(calendar.items()):
        if not first_date <= holiday_date <= last_date:
            continue
        date_text = holiday_date.isoformat()
        holiday_weekday = pd.Timestamp(holiday_date).day_name()
        holiday_records = int(
            (
                data["loc_country_code"].eq(country_code) & data["date"].eq(date_text)
            ).sum()
        )
        controls = time_daily_country.loc[
            time_daily_country["loc_country_code"].eq(country_code)
            & time_daily_country["weekday"].eq(holiday_weekday)
            & time_daily_country["is_holiday"].ne(True)
            & time_daily_country["date"].lt(date_text),
            "records",
        ]
        holiday_rows.append(
            {
                "loc_country_code": country_code,
                "date": date_text,
                "holiday_name": holiday_name,
                "holiday_records": holiday_records,
                "control_mean": controls.mean(),
            }
        )

time_holidays = pd.DataFrame(holiday_rows)
time_holidays.to_csv(OUTPUT / "time_holidays.csv", index=False, encoding="utf-8-sig")
display(time_holidays)


### 3.2.2 当地日期记录数

In [ ]:
x = np.arange(len(time_daily_local))
bar_colors = ["#BBBBBB"] + [colors[0]] * (len(x) - 2) + ["#BBBBBB"]
fig, ax = plt.subplots(figsize=(13, 7.5))
fig.subplots_adjust(left=0.09, right=0.95, top=0.86, bottom=0.13)
fig.suptitle(
    "当地日期记录数" if LANG == "zh" else "Records by local date",
    fontsize=20,
    weight="bold",
    color="#17364B",
)
ax.bar(x, time_daily_local["records"], color=bar_colors)
ax.set_xticks(x[::3], [date[5:] for date in time_daily_local["date"][::3]])
ax.set_ylabel("记录数" if LANG == "zh" else "Records")
for edge in [0, len(x) - 1]:
    ax.annotate(
        "不完整" if LANG == "zh" else "Partial",
        (edge, time_daily_local.iloc[edge]["records"]),
        xytext=(0, 10),
        textcoords="offset points",
        ha="center",
    )
fig.savefig(time_charts / f"01_local_dates_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![当地日期记录数](outputs/simple/charts/time/01_local_dates_zh.png)

### 3.2.3 星期分布

In [ ]:
fig, axes = plt.subplots(figsize=(13, 7.5), ncols=3)
fig.subplots_adjust(left=0.09, right=0.95, top=0.86, bottom=0.13, wspace=0.35)
fig.suptitle(
    "各星期的日均值与中位数" if LANG == "zh" else "Daily mean and median by weekday",
    fontsize=20,
    weight="bold",
    color="#17364B",
)
weekday_tick_labels = (
    list("一二三四五六日") if LANG == "zh" else ["M", "Tu", "W", "Th", "F", "Sa", "Su"]
)
for ax, country_code in zip(axes, ["JP", "ID", "KR"]):
    weekday_part = time_weekday.loc[time_weekday["loc_country_code"].eq(country_code)]
    ax.plot(
        weekday_part["weekday_number"],
        weekday_part["daily_mean"],
        "o-",
        color=colors[1],
        label="日均值" if LANG == "zh" else "Mean",
    )
    ax.plot(
        weekday_part["weekday_number"],
        weekday_part["daily_median"],
        "s--",
        color=colors[0],
        label="日中位数" if LANG == "zh" else "Median",
    )
    ax.set_xticks(range(7), weekday_tick_labels)
    ax.set_title(country_code_labels[country_code])
    ax.set_ylim(bottom=0)
    ax.set_ylabel("每天记录数" if LANG == "zh" else "Records per day")
    ax.legend(frameon=False)
fig.savefig(time_charts / f"02_weekday_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![星期分布](outputs/simple/charts/time/02_weekday_zh.png)

### 3.2.4 当地小时分布

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7.5))
fig.subplots_adjust(left=0.09, right=0.95, top=0.86, bottom=0.13)
fig.suptitle(
    "各市场当地小时分布" if LANG == "zh" else "Local-hour distribution by market",
    fontsize=20,
    weight="bold",
    color="#17364B",
)
for country_code, market_color in zip(["JP", "ID", "KR", "VN", "TH"], colors):
    hourly_part = time_hourly.loc[time_hourly["loc_country_code"].eq(country_code)]
    ax.plot(
        hourly_part["hour"],
        hourly_part["country_hour_share"] * 100,
        label=country_code_labels[country_code],
        color=market_color,
        linewidth=2,
    )
ax.set_xticks(range(0, 24, 2))
ax.set_xlim(0, 23)
ax.set_ylim(bottom=0)
ax.set_xlabel("当地小时" if LANG == "zh" else "Local hour")
ax.set_ylabel(
    "市场内记录占比（%）" if LANG == "zh" else "Within-market record share (%)"
)
ax.legend(frameon=False, ncols=3)
fig.savefig(time_charts / f"03_local_hours_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![当地小时分布](outputs/simple/charts/time/03_local_hours_zh.png)

### 3.2.5 星期与小时分布

In [ ]:
fig, axes = plt.subplots(figsize=(13, 7.5), ncols=3)
fig.subplots_adjust(left=0.09, right=0.95, top=0.86, bottom=0.13, wspace=0.50)
fig.suptitle(
    "星期与小时分布" if LANG == "zh" else "Weekday and hour distribution",
    fontsize=20,
    weight="bold",
    color="#17364B",
)
weekday_tick_labels = (
    list("一二三四五六日")
    if LANG == "zh"
    else ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
)
for ax, country_code in zip(axes, ["JP", "ID", "KR"]):
    heatmap_data = (
        time_weekday_hour.loc[time_weekday_hour["loc_country_code"].eq(country_code)]
        .pivot(index="weekday_number", columns="hour", values="daily_mean")
        .reindex(index=range(7), columns=range(24), fill_value=0)
    )
    image = ax.imshow(heatmap_data, aspect="auto", cmap="YlOrRd", vmin=0)
    ax.set_title(country_code_labels[country_code])
    ax.set_xticks([0, 6, 12, 18, 23])
    ax.set_yticks(range(7), weekday_tick_labels)
    ax.set_xlabel("当地小时" if LANG == "zh" else "Local hour")
    fig.colorbar(image, ax=ax, shrink=0.70)
fig.savefig(time_charts / f"04_weekday_hour_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![星期与小时分布](outputs/simple/charts/time/04_weekday_hour_zh.png)

### 3.2.6 节日与对照日

In [ ]:
fig, axes = plt.subplots(figsize=(13, 7.5), ncols=3)
fig.subplots_adjust(left=0.09, right=0.95, top=0.86, bottom=0.13, wspace=0.40)
fig.suptitle(
    "节日与对照日记录数" if LANG == "zh" else "Holiday and control-day records",
    fontsize=20,
    weight="bold",
    color="#17364B",
)
for ax, country_code in zip(axes, ["ID", "SG", "HK"]):
    holiday_part = time_holidays.loc[time_holidays["loc_country_code"].eq(country_code)]
    x = np.arange(len(holiday_part))
    ax.bar(
        x - 0.18,
        holiday_part["holiday_records"],
        width=0.35,
        color=colors[1],
        label="节日当天" if LANG == "zh" else "Holiday",
    )
    ax.bar(
        x + 0.18,
        holiday_part["control_mean"],
        width=0.35,
        color=colors[0],
        label="同星期对照" if LANG == "zh" else "Matched weekday",
    )
    ax.set_xticks(x, [date[5:] for date in holiday_part["date"]])
    ax.set_title(country_code_labels[country_code])
    ax.set_ylabel("记录数" if LANG == "zh" else "Records")
    ax.legend(frameon=False)
fig.savefig(time_charts / f"06_holidays_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![节日与对照日](outputs/simple/charts/time/06_holidays_zh.png)

## 3.3 IP Address

### 3.3.1 汇总表

In [ ]:
scope_categories = [
    "Global unicast candidate",
    "Reserved (library)",
    "Multicast",
    "Other special",
]
scope_group = np.select(
    [
        data["ip_is_multicast"].fillna(False),
        data["ip_is_reserved"].fillna(False),
        data["ip_is_global"].fillna(False),
    ],
    [scope_categories[2], scope_categories[1], scope_categories[0]],
    default="Other special",
)
ip_scope = (
    data.assign(ip_scope_category=scope_group)
    .groupby(["ip_version", "ip_scope_category"])
    .size()
    .rename("records")
    .reset_index()
)

ip_countries = (
    data.dropna(subset=["ip_country_en"])
    .groupby(["ip_country_en", "ip_country_zh"])
    .size()
    .rename("records")
    .reset_index()
    .sort_values("records", ascending=False)
)
ip_countries["matched_share"] = ip_countries["records"] / ip_countries["records"].sum()

ip_asns = (
    data.dropna(subset=["ip_asn"])
    .groupby(["ip_asn", "ip_asn_name", "ip_asn_organization"], dropna=False)
    .size()
    .rename("records")
    .reset_index()
    .sort_values("records", ascending=False)
)
ip_asns["matched_share"] = ip_asns["records"] / ip_asns["records"].sum()


In [ ]:
location_totals = (
    data.dropna(subset=["loc_country_en"])
    .groupby(["loc_country_en", "loc_country_zh"])
    .size()
    .rename("records")
    .reset_index()
)
comparable_ip = data.dropna(subset=["loc_country_en", "ip_country_en"]).assign(
    same_country=data["loc_country_en"].eq(data["ip_country_en"])
)
ip_agreement = (
    comparable_ip.groupby(["loc_country_en", "loc_country_zh"])
    .agg(comparable=("user_id", "size"), same_country=("same_country", "sum"))
    .reset_index()
    .merge(
        location_totals,
        on=["loc_country_en", "loc_country_zh"],
        how="right",
        validate="one_to_one",
    )
)
ip_agreement[["comparable", "same_country"]] = ip_agreement[
    ["comparable", "same_country"]
].fillna(0)
ip_agreement["same_country_rate"] = ip_agreement["same_country"] / ip_agreement[
    "comparable"
].replace(0, np.nan)
ip_agreement = ip_agreement.sort_values("records", ascending=False)

for table_name, table in {
    "ip_scope": ip_scope,
    "ip_countries": ip_countries,
    "ip_asns": ip_asns,
    "ip_agreement": ip_agreement,
}.items():
    table.to_csv(OUTPUT / f"{table_name}.csv", index=False, encoding="utf-8-sig")

display(ip_scope)
display(ip_agreement.head(10))


### 3.3.2 IP 地址类型

In [ ]:
scope_pivot = (
    ip_scope.pivot(index="ip_version", columns="ip_scope_category", values="records")
    .reindex(index=[4, 6], columns=scope_categories, fill_value=0)
    .fillna(0)
)
scope_share = scope_pivot.div(scope_pivot.sum(axis=1), axis=0) * 100
scope_labels_zh = {
    "Global unicast candidate": "公网单播候选",
    "Reserved (library)": "保留地址",
    "Multicast": "组播",
    "Other special": "其他特殊地址",
}

fig, ax = plt.subplots(figsize=(13, 7.5))
fig.subplots_adjust(left=0.22, right=0.91, top=0.86, bottom=0.15)
fig.suptitle(
    "IP 地址类型" if LANG == "zh" else "IP address types",
    fontsize=19,
    weight="bold",
    color="#17364B",
)
left = np.zeros(2)
scope_colors = [colors[0], colors[1], colors[2], "#AAAAAA"]
for category, category_color in zip(scope_categories, scope_colors):
    values = scope_share[category].to_numpy()
    label = scope_labels_zh[category] if LANG == "zh" else category
    ax.barh(["IPv4", "IPv6"], values, left=left, label=label, color=category_color)
    for row_number, value in enumerate(values):
        if value > 10:
            ax.text(
                left[row_number] + value / 2,
                row_number,
                f"{value:.1f}%",
                ha="center",
                va="center",
                color="white",
            )
    left += values
ax.set_xlim(0, 100)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncols=2, frameon=False)
fig.savefig(ip_charts / f"01_scope_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![IP 地址类型](outputs/simple/charts/ip/01_scope_zh.png)

### 3.3.3 IP 国家／地区

In [ ]:
country_part = ip_countries.head(10).sort_values("matched_share")
country_names = (
    country_part["ip_country_zh"].fillna(country_part["ip_country_en"])
    if LANG == "zh"
    else country_part["ip_country_en"]
)
country_values = country_part["matched_share"] * 100

fig, ax = plt.subplots(figsize=(13, 7.5))
fig.subplots_adjust(left=0.22, right=0.91, top=0.86, bottom=0.13)
fig.suptitle(
    (
        "IP 国家／地区分布（2022 年 12 月）"
        if LANG == "zh"
        else "IP countries/regions (December 2022)"
    ),
    fontsize=19,
    weight="bold",
    color="#17364B",
)
ax.barh(country_names, country_values, color=colors[0])
ax.set_xlim(0, country_values.max() * 1.22)
for row_number, value in enumerate(country_values):
    ax.text(
        value + country_values.max() * 0.015, row_number, f"{value:.2f}%", va="center"
    )
fig.savefig(ip_charts / f"02_countries_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![IP 国家／地区](outputs/simple/charts/ip/02_countries_zh.png)

### 3.3.4 BGP 起源 ASN

In [ ]:
asn_part = ip_asns.head(8).sort_values("matched_share")
asn_names = asn_part["ip_asn_name"].fillna(asn_part["ip_asn_organization"])
asn_labels = (
    "AS"
    + asn_part["ip_asn"].astype(int).astype(str)
    + " · "
    + asn_names.fillna("Unknown").str.slice(0, 42)
)
asn_values = asn_part["matched_share"] * 100

fig, ax = plt.subplots(figsize=(13, 7.5))
fig.subplots_adjust(left=0.39, right=0.91, top=0.86, bottom=0.13)
fig.suptitle(
    (
        "BGP 起源 ASN（2022 年 12 月）"
        if LANG == "zh"
        else "BGP origin ASNs (December 2022)"
    ),
    fontsize=19,
    weight="bold",
    color="#17364B",
)
ax.barh(asn_labels, asn_values, color=colors[0])
ax.set_xlim(0, asn_values.max() * 1.22)
for row_number, value in enumerate(asn_values):
    ax.text(value + asn_values.max() * 0.015, row_number, f"{value:.2f}%", va="center")
fig.savefig(ip_charts / f"03_asns_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![BGP 起源 ASN](outputs/simple/charts/ip/03_asns_zh.png)

### 3.3.5 IP 与坐标国家一致率

In [ ]:
agreement_part = (
    ip_agreement.loc[ip_agreement["comparable"].gt(0)]
    .head(10)
    .sort_values("same_country_rate")
)
agreement_names = (
    agreement_part["loc_country_zh"].fillna(agreement_part["loc_country_en"])
    if LANG == "zh"
    else agreement_part["loc_country_en"]
)
agreement_values = agreement_part["same_country_rate"] * 100

fig, ax = plt.subplots(figsize=(13, 7.5))
fig.subplots_adjust(left=0.22, right=0.91, top=0.86, bottom=0.13)
fig.suptitle(
    (
        "历史 IP 与坐标的国家／地区一致率"
        if LANG == "zh"
        else "Historical IP and coordinate-country agreement"
    ),
    fontsize=19,
    weight="bold",
    color="#17364B",
)
ax.barh(agreement_names, agreement_values, color=colors[0])
ax.set_xlim(0, agreement_values.max() * 1.22)
ax.set_xlabel(
    "可比较记录中的一致比例（%）"
    if LANG == "zh"
    else "Agreement among comparable records (%)"
)
for row_number, value in enumerate(agreement_values):
    ax.text(
        value + agreement_values.max() * 0.015, row_number, f"{value:.2f}%", va="center"
    )
fig.savefig(ip_charts / f"04_agreement_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![IP 与坐标国家一致率](outputs/simple/charts/ip/04_agreement_zh.png)

## 3.4 User Agent

### 3.4.1 操作系统占比

In [ ]:
ua_os_share = (
    data.assign(os_group=data["os"].fillna("Unresolved / missing"))
    .groupby("os_group")
    .size()
    .rename("records")
    .reset_index()
)
ua_os_share.to_csv(OUTPUT / "ua_os_share.csv", index=False, encoding="utf-8-sig")

os_order = ["Android", "iOS", "Mac OS X", "Unresolved / missing"]
os_share_part = (
    ua_os_share.set_index("os_group").reindex(os_order, fill_value=0).reset_index()
)
os_share_part["share"] = os_share_part["records"] / len(data) * 100
os_labels = [
    "Android",
    "iOS",
    "Mac OS X",
    "未确定／UA 缺失" if LANG == "zh" else "Unresolved / missing UA",
]

fig, ax = plt.subplots(figsize=(12, 6.5))
fig.subplots_adjust(left=0.22, right=0.94, top=0.84, bottom=0.13)
fig.suptitle(
    "操作系统占比" if LANG == "zh" else "Operating-system share",
    fontsize=22,
    weight="bold",
    color="#17364B",
)
bars = ax.barh(
    range(4),
    os_share_part["share"],
    color=[colors[0], colors[2], colors[1], "#AAAAAA"],
    height=0.52,
)
ax.set_yticks(range(4), os_labels)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xticks(range(0, 101, 20), [f"{value}%" for value in range(0, 101, 20)])
ax.grid(axis="x", alpha=0.15)
ax.set_axisbelow(True)
for row_number, row in os_share_part.iterrows():
    share = row["share"]
    label = f"{share:.3f}% · {int(row['records']):,}"
    if share > 70:
        ax.text(
            share - 2,
            row_number,
            label,
            ha="right",
            va="center",
            color="white",
            weight="bold",
        )
    else:
        ax.text(share + 1.5, row_number, label, ha="left", va="center")
fig.savefig(ua_charts / f"08_os_share_{suffix}.png", dpi=180, facecolor="white")
plt.close(fig)
print(f"Saved 19 {suffix} charts to {CHARTS}")


![操作系统占比](outputs/simple/charts/ua/08_os_share_zh.png)

### 3.4.2 汇总表

In [ ]:
ua_chart_rows = []
for raw_ua in data["user_agent"].dropna().drop_duplicates():
    parsed_ua = parse(raw_ua)
    browser_family = getattr(parsed_ua.user_agent, "family", None)
    if browser_family in [None, "Other"]:
        browser_family = pd.NA
    chrome_match = re.search(r"\bChrome/(\d+)", raw_ua)
    ua_chart_rows.append(
        {
            "user_agent": raw_ua,
            "browser_family": browser_family,
            "chrome_major": int(chrome_match.group(1)) if chrome_match else pd.NA,
        }
    )

ua_chart_lookup = pd.DataFrame(ua_chart_rows)
ua_plot_data = data.merge(
    ua_chart_lookup,
    on="user_agent",
    how="left",
    validate="many_to_one",
    sort=False,
)
missing_ua = ua_plot_data["user_agent"].isna()
dalvik_token = ua_plot_data["user_agent"].str.startswith("Dalvik/", na=False)
explicit_wv = ua_plot_data["user_agent"].str.contains(
    r"(?:[;(]\s*)wv(?:[;)])", regex=True, na=False
)
parser_webview = ua_plot_data["browser_family"].isin(
    ["Chrome Mobile WebView", "Mobile Safari UI/WKWebView"]
)
unresolved_client = ua_plot_data["browser_family"].isna()
ua_plot_data["client_context"] = np.select(
    [missing_ua, dalvik_token, explicit_wv, parser_webview, unresolved_client],
    [
        "Missing UA",
        "Dalvik token",
        "Explicit Android wv",
        "Parser WebView only",
        "Unresolved client",
    ],
    default="Other parsed client",
)


In [ ]:
ua_plot_data["os_major"] = pd.to_numeric(
    ua_plot_data["os_version"].str.extract(r"^(\d+)", expand=False),
    errors="coerce",
)
ua_os_versions = (
    ua_plot_data.groupby(["os", "os_major"], dropna=False)
    .size()
    .rename("records")
    .reset_index()
)
ua_os_versions["within_os_share"] = ua_os_versions["records"] / ua_os_versions.groupby(
    "os", dropna=False
)["records"].transform("sum")

ua_context = (
    ua_plot_data.groupby("client_context").size().rename("records").reset_index()
)
ua_context["sample_share"] = ua_context["records"] / len(ua_plot_data)

ua_plot_data["brand_chart"] = ua_plot_data["device_brand"].mask(
    ua_plot_data["device_brand"].str.startswith("Generic_", na=False)
)
ua_brands = (
    ua_plot_data.groupby("brand_chart", dropna=False)
    .size()
    .rename("records")
    .reset_index()
    .sort_values("records", ascending=False)
)
ua_brands["sample_share"] = ua_brands["records"] / len(ua_plot_data)
ua_models = (
    ua_plot_data.groupby(["brand_chart", "device_model"], dropna=False)
    .size()
    .rename("records")
    .reset_index()
    .sort_values("records", ascending=False)
)
ua_models["sample_share"] = ua_models["records"] / len(ua_plot_data)


In [ ]:
ua_country_brand = (
    ua_plot_data.dropna(subset=["loc_country_code"])
    .groupby(["loc_country_code", "brand_chart"], dropna=False)
    .size()
    .rename("records")
    .reset_index()
)

webview_rows = ua_plot_data.loc[
    ua_plot_data["browser_family"].eq("Chrome Mobile WebView")
].copy()
webview_rows["event_date_utc"] = webview_rows["timestamp_utc"].dt.strftime("%Y-%m-%d")
webview_rows["version_group"] = (
    webview_rows["chrome_major"].astype("Int64").astype("string")
)
webview_rows["version_group"] = webview_rows["version_group"].where(
    webview_rows["chrome_major"].isin([107, 108]), "Other / unknown"
)
ua_webview_versions = (
    webview_rows.groupby(["event_date_utc", "version_group"])
    .size()
    .rename("records")
    .reset_index()
)
ua_webview_versions["day_share"] = ua_webview_versions["records"] / (
    ua_webview_versions.groupby("event_date_utc")["records"].transform("sum")
)

for table_name, table in {
    "ua_os_versions": ua_os_versions,
    "ua_context": ua_context,
    "ua_brands": ua_brands,
    "ua_models": ua_models,
    "ua_country_brand": ua_country_brand,
    "ua_webview_versions": ua_webview_versions,
}.items():
    table.to_csv(OUTPUT / f"{table_name}.csv", index=False, encoding="utf-8-sig")

display(ua_context.sort_values("records", ascending=False))


### 3.4.3 操作系统版本

In [ ]:
fig, axes = plt.subplots(figsize=(13, 7.5), ncols=2)
fig.subplots_adjust(left=0.10, right=0.95, top=0.86, bottom=0.13, wspace=0.50)
fig.suptitle(
    "操作系统版本分布" if LANG == "zh" else "Operating-system version distribution",
    fontsize=19,
    weight="bold",
    color="#17364B",
)
for ax, os_name in zip(axes, ["Android", "iOS"]):
    version_part = ua_os_versions.loc[
        ua_os_versions["os"].eq(os_name) & ua_os_versions["os_major"].notna()
    ].sort_values("os_major")
    version_labels = version_part["os_major"].astype(int).astype(str)
    ax.bar(
        version_labels,
        version_part["within_os_share"] * 100,
        color=colors[0],
    )
    ax.set_title(os_name)
    ax.set_ylabel("系统内占比（%）" if LANG == "zh" else "Within-OS share (%)")
fig.savefig(ua_charts / f"02_os_versions_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![操作系统版本](outputs/simple/charts/ua/02_os_versions_zh.png)

### 3.4.4 客户端环境

In [ ]:
context_order = [
    "Explicit Android wv",
    "Dalvik token",
    "Parser WebView only",
    "Missing UA",
    "Other parsed client",
    "Unresolved client",
]
context_labels_zh = {
    "Explicit Android wv": "明确 Android wv 标记",
    "Dalvik token": "Dalvik 标记",
    "Parser WebView only": "仅解析器判为 WebView",
    "Missing UA": "UA 缺失",
    "Other parsed client": "其他已解析客户端",
    "Unresolved client": "客户端未确定",
}
context_part = (
    ua_context.set_index("client_context")
    .reindex(context_order, fill_value=0)
    .reset_index()
    .sort_values("sample_share")
)
context_labels = (
    context_part["client_context"].map(context_labels_zh)
    if LANG == "zh"
    else context_part["client_context"]
)
context_values = context_part["sample_share"] * 100

fig, ax = plt.subplots(figsize=(13, 7.5))
fig.subplots_adjust(left=0.25, right=0.91, top=0.86, bottom=0.13)
fig.suptitle(
    "客户端环境" if LANG == "zh" else "Client context",
    fontsize=19,
    weight="bold",
    color="#17364B",
)
ax.barh(context_labels, context_values, color=colors[0])
ax.set_xlim(0, context_values.max() * 1.22)
for row_number, value in enumerate(context_values):
    text_value = "<0.01%" if 0 < value < 0.01 else f"{value:.2f}%"
    ax.text(value + context_values.max() * 0.015, row_number, text_value, va="center")
fig.savefig(ua_charts / f"03_client_context_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![客户端环境](outputs/simple/charts/ua/03_client_context_zh.png)

### 3.4.5 设备品牌与型号

In [ ]:
brand_part = (
    ua_brands.dropna(subset=["brand_chart"]).head(6).sort_values("sample_share")
)
model_part = (
    ua_models.dropna(subset=["device_model"]).head(8).sort_values("sample_share")
)
fig, axes = plt.subplots(figsize=(13, 7.5), ncols=2)
fig.subplots_adjust(left=0.12, right=0.95, top=0.86, bottom=0.13, wspace=0.85)
fig.suptitle(
    "设备品牌与型号" if LANG == "zh" else "Device brands and models",
    fontsize=19,
    weight="bold",
    color="#17364B",
)
for ax, labels, values in [
    (axes[0], brand_part["brand_chart"], brand_part["sample_share"] * 100),
    (axes[1], model_part["device_model"], model_part["sample_share"] * 100),
]:
    ax.barh(labels, values, color=colors[0])
    ax.set_xlim(0, values.max() * 1.22)
    for row_number, value in enumerate(values):
        ax.text(value + values.max() * 0.015, row_number, f"{value:.2f}%", va="center")
fig.savefig(ua_charts / f"04_devices_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![设备品牌与型号](outputs/simple/charts/ua/04_devices_zh.png)

### 3.4.6 各市场设备品牌

In [ ]:
market_order = ["JP", "ID", "KR", "VN", "TH"]
brand_groups = ["Samsung", "Motorola", "Google", "Apple", "Other named", "Unknown"]
country_brand_plot = ua_country_brand.copy()
country_brand_plot["brand_group"] = country_brand_plot["brand_chart"].fillna("Unknown")
country_brand_plot.loc[
    ~country_brand_plot["brand_group"].isin(brand_groups), "brand_group"
] = "Other named"
brand_pivot = (
    country_brand_plot.groupby(["loc_country_code", "brand_group"])["records"]
    .sum()
    .unstack(fill_value=0)
    .reindex(index=market_order, columns=brand_groups, fill_value=0)
)
brand_share = brand_pivot.div(brand_pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(13, 7.5))
fig.subplots_adjust(left=0.15, right=0.95, top=0.83, bottom=0.13)
fig.suptitle(
    "各市场设备品牌构成" if LANG == "zh" else "Device-brand composition by market",
    fontsize=19,
    weight="bold",
    color="#17364B",
)
left = np.zeros(len(market_order))
for brand_group, brand_color in zip(brand_groups, colors):
    label = brand_group
    if LANG == "zh":
        label = {"Other named": "其他具名品牌", "Unknown": "品牌未知"}.get(
            brand_group, brand_group
        )
    ax.barh(
        range(len(market_order)),
        brand_share[brand_group],
        left=left,
        label=label,
        color=brand_color,
    )
    left += brand_share[brand_group].to_numpy()
ax.set_yticks(
    range(len(market_order)),
    [country_code_labels[code] for code in market_order],
)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("市场内占比（%）" if LANG == "zh" else "Within-market share (%)")
ax.legend(ncols=3, loc="lower center", bbox_to_anchor=(0.5, 1.01), frameon=False)
fig.savefig(ua_charts / f"05_country_brands_{suffix}.png", dpi=170, facecolor="white")
plt.close(fig)


![各市场设备品牌](outputs/simple/charts/ua/05_country_brands_zh.png)